# SD-004 — Synthetic Retail Demand Data Generator
This notebook creates a fictional retail dataset for demand-planning practice. It exports five required tables and one supporting calendar table. Run the cells from top to bottom.



## 1. Configure the example
Change `N_DAYS` (minimum 120) to generate a longer date range. The fixed seed makes each configuration reproducible. All tunable simulation assumptions are collected in the next code cell so adjustments are easy to review.



In [66]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
N_DAYS = 120
START_DATE = "2025-01-01"
WEEKDAY_FACTORS = [1.000, 0.925, 0.908, 0.911, 1.032, 1.247, 1.243]
HOLIDAY_FACTOR, PROMOTION_PROBABILITY = 1.35, 0.04
PROMOTION_DISCOUNTS, FORCED_PROMOTION_DISCOUNT = [0.10, 0.15, 0.20, 0.25], 0.50
PRICE_VARIATION, PROMOTION_DEMAND_EFFECT = (0.98, 1.02), 1.4
DEMAND_TREND, INVENTORY_MULTIPLIER, RANDOM_SHORTAGE_PROBABILITY = 0.12, 2.0, 0.003
FORCED_STOCKOUT = ("2025-04-25", "S001", "P001")
FORCED_PROMOTION = ("2025-02-14", "S003", "P002")



## 2. Create the store and product tables
These small dimension tables define the fictional stores and products used by the daily records.



## 3. Create the calendar and calibrated weekday factors
Weekday multipliers use rounded ratios from a seeded sample of 1,000 M5 item-store series; these inform weekday shape only.



## 4. Define the daily demand process
Expected demand combines product baseline, store scale, weekday pattern, trend, holiday, and price/promotion assumptions. Random draws add day-to-day variation.



In [67]:
def generate_tables():
    assert N_DAYS >= 120, "At least 120 days are needed for the fixed edge cases."
    rng = np.random.default_rng(SEED)
    dates = pd.date_range(START_DATE, periods=N_DAYS, freq="D")
    stores = pd.DataFrame([
        ("S001", "North Market", "North", 0.82),
        ("S002", "Riverside Market", "East", 0.94),
        ("S003", "Central Market", "Central", 1.00),
        ("S004", "Hilltop Market", "West", 1.13),
        ("S005", "South Market", "South", 1.25),
    ], columns=["store_id", "store_name", "region", "demand_index"])
    products = pd.DataFrame([
        ("P001", "Apples", "Produce", 2.49, 18),
        ("P002", "Milk", "Dairy", 3.29, 15),
        ("P003", "Coffee", "Beverages", 8.99, 9),
        ("P004", "Chips", "Snacks", 3.79, 12),
        ("P005", "Laundry Detergent", "Household", 10.99, 5),
    ], columns=["product_id", "product_name", "category", "base_price", "base_daily_demand"])
    holiday_names = {"2025-02-14": "Valentines Day", "2025-04-20": "Easter"}
    calendar = pd.DataFrame({"date": dates.strftime("%Y-%m-%d"), "day_of_week": dates.day_of_week, "is_weekend": dates.day_of_week >= 5})
    calendar["holiday_name"] = calendar["date"].map(holiday_names).fillna("")
    calendar["is_holiday"] = calendar["holiday_name"] != ""

    sales_rows, promotion_rows, inventory_rows = [], [], []
    for store in stores.itertuples(index=False):
        for product in products.itertuples(index=False):
            for day_number, date in enumerate(dates):
                date_text = date.strftime("%Y-%m-%d")
                forced_promo = (date_text, store.store_id, product.product_id) == FORCED_PROMOTION
                forced_stockout = (date_text, store.store_id, product.product_id) == FORCED_STOCKOUT
                is_promotion = forced_promo or rng.random() < PROMOTION_PROBABILITY
                discount = FORCED_PROMOTION_DISCOUNT if forced_promo else (float(rng.choice(PROMOTION_DISCOUNTS)) if is_promotion else 0.0)
                regular_price = product.base_price * rng.uniform(*PRICE_VARIATION)
                unit_price = round(regular_price * (1 - discount), 2)
                holiday = HOLIDAY_FACTOR if date_text in holiday_names else 1.0
                trend = 1 + DEMAND_TREND * day_number / (N_DAYS - 1)
                price_effect = (regular_price / unit_price) ** 0.7
                promo_effect = 1 + PROMOTION_DEMAND_EFFECT * discount
                expected = product.base_daily_demand * store.demand_index * WEEKDAY_FACTORS[date.day_of_week] * trend * holiday * price_effect * promo_effect
                demand = int(rng.poisson(expected))
                available = max(demand, int(rng.poisson(max(expected * INVENTORY_MULTIPLIER, 1)) + 2))
                if rng.random() < RANDOM_SHORTAGE_PROBABILITY and demand > 0:
                    available = int(rng.integers(0, demand))
                if forced_stockout:
                    available = 0
                sold = min(demand, available)
                edge = "forced_stockout" if forced_stockout else ("promo_peak" if forced_promo else "")
                key = {"date": date_text, "store_id": store.store_id, "product_id": product.product_id}
                promotion_rows.append({**key, "unit_price": unit_price, "discount_pct": discount, "is_promotion": is_promotion})
                inventory_rows.append({**key, "available_units": available, "closing_units": available - sold, "stockout_flag": available < demand})
                sales_rows.append({**key, "units_sold": sold, "edge_case": edge})
    return {
        "stores": stores,
        "products": products,
        "calendar": calendar,
        "daily_sales": pd.DataFrame(sales_rows),
        "promotions": pd.DataFrame(promotion_rows),
        "inventory_snapshots": pd.DataFrame(inventory_rows),
    }



## 5. Generate the daily tables
Each store-product-date combination creates one row in each daily table. Latent demand is used to simulate sales but is not exported.



In [68]:
def validate_tables(tables):
    key = ["date", "store_id", "product_id"]
    stores, products, calendar = tables["stores"], tables["products"], tables["calendar"]
    sales, promotions, inventory = (tables[name] for name in ["daily_sales", "promotions", "inventory_snapshots"])
    assert stores.store_id.is_unique and products.product_id.is_unique and calendar.date.is_unique
    assert len(sales) == len(stores) * len(products) * len(calendar)
    for table in [sales, promotions, inventory]:
        assert not table.duplicated(key).any()
    assert set(sales.store_id) <= set(stores.store_id)
    assert set(sales.product_id) <= set(products.product_id)
    assert set(sales.date) <= set(calendar.date)
    assert (sales.units_sold >= 0).all() and (promotions.unit_price > 0).all()
    assert promotions.discount_pct.between(0, 0.5).all()
    assert (sales.units_sold <= inventory.available_units).all()
    assert (inventory.closing_units == inventory.available_units - sales.units_sold).all()
    assert sales.edge_case.eq("forced_stockout").sum() == 1
    assert sales.edge_case.eq("promo_peak").sum() == 1



## 6. Assemble and validate the tables
The daily tables share a composite key. These checks confirm the schema, links, value ranges, inventory arithmetic, and deliberate cases.



In [69]:
if __name__ == "__main__":
    tables = generate_tables()
    validate_tables(tables)
    output_dir = Path(__file__).resolve().parents[2] / "Sprint1" / "data" if "__file__" in globals() and Path(__file__).suffix != ".ipynb" else Path("../data")
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(output_dir / f"{name}.csv", index=False)
    table_counts = pd.DataFrame({"table": tables.keys(), "rows": [len(table) for table in tables.values()]})
    print("Primary-key, relationship, range, inventory, and edge-case checks passed.")
    print(table_counts.to_string(index=False))


Primary-key, relationship, range, inventory, and edge-case checks passed.
              table  rows
             stores     5
           products     5
           calendar   120
        daily_sales  3000
         promotions  3000
inventory_snapshots  3000
